# Artwork Multi-Method Eval — EA, KVzip, Finch (CPT vs Full)
Results + checkpoint saved to **Google Drive**.

**Methods included:**
- `ExpectedAttentionPress`
- `KVzipPress`
- `FinchPress` (Running two variants: Full Context vs CPT)


## Step 1-3: Setup (Standard)

In [ ]:
# Run this once per session
import subprocess, sys, os
from pathlib import Path

# Clone
BENCH_URL = "https://github.com/physical168/kv-compression-benchmark.git"
CE_URL    = "https://github.com/GabrieleSanmartino/CompressionExperiments.git"
os.chdir("/content")
subprocess.run(["git","clone","--depth","1",BENCH_URL,"/content/kv-compression-benchmark"])
subprocess.run(["git","clone","--depth","1",CE_URL,"/content/CompressionExperiments"])

# Install
_ce = Path("/content/CompressionExperiments")
subprocess.check_call([sys.executable,"-m","pip","install","-q","-U","pip"])
subprocess.check_call([sys.executable,"-m","pip","install","-q",
    "transformers>=5.0","accelerate","bitsandbytes","pandas","pyyaml","scikit-learn","matplotlib","tqdm","pillow","git-lfs"])

# Git LFS pull for images
os.chdir("/content/CompressionExperiments")
subprocess.run(["git","lfs","install"])
subprocess.run(["git","lfs","pull"])

_kvp = _ce / "kvpress"
subprocess.check_call([sys.executable,"-m","pip","install",str(_kvp)])
import kvpress
print("kvpress installed:", kvpress.__file__)


## Step 1.5 — Mount Google Drive

In [ ]:
from google.colab import drive
import os, pandas as pd
from pathlib import Path
drive.mount("/content/drive", force_remount=False)
DRIVE_OUT        = "/content/drive/MyDrive/kv-compression-benchmark/ce_artwork_all_results"
Path(DRIVE_OUT).mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH  = Path(DRIVE_OUT) / "checkpoint_all.csv"
RESULTS_ROOT     = Path(DRIVE_OUT) / "results"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)


## Step 2 — Prepare dataset

In [ ]:
import shutil
from pathlib import Path
CE_ART = Path("/content/CompressionExperiments/experiment_manager/datasets/artwork")
IMAGES_DIR = CE_ART / "images"
DATASET_PATH = CE_ART / "paintings.csv"
# Ensure CSV is present
if not DATASET_PATH.is_file():
    shutil.copy2("/content/kv-compression-benchmark/paintings.csv", DATASET_PATH)
print("Images count:", len(list(IMAGES_DIR.glob("*"))))


## Step 3 — Load model

In [ ]:
import torch, sys, importlib.util
from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration, BitsAndBytesConfig
MODEL_ID = "llava-hf/llama3-llava-next-8b-hf"
bnb_cfg = BitsAndBytesConfig(load_in_8bit=True)
processor = LlavaNextProcessor.from_pretrained(MODEL_ID)
model = LlavaNextForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto", quantization_config=bnb_cfg)

# Patch for kvpress
_patch = "/content/kv-compression-benchmark/benchmarks/artwork_eval/llava_kvpress_patch.py"
_spec = importlib.util.spec_from_file_location("patch", _patch)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
_mod.apply_kvpress_compatibility_patches(model)


## Step 4 — Configuration (Methods & Ratios)

In [ ]:
from kvpress import ExpectedAttentionPress, KVzipPress, FinchPress

MAX_ROWS = 10
COMPRESSION_RATIOS = [0.4, 0.8, 0.95]

# method_key -> (PressClass, use_cpt)
EXPERIMENT_CONFIGS = {
    "EA": (ExpectedAttentionPress, True),
    "KVzip": (KVzipPress, True),
    "Finch_Full": (FinchPress, False),
    "Finch_CPT": (FinchPress, True),
}

_EXTRACT_SUFFIX = " (be concise, no explanation, no introductory text, just the answer) ?"

def build_prompt(row, question, is_boolean, use_cpt):
    if use_cpt:
        # CPT version: No metadata
        context = ""
    else:
        # Full Context version: Add metadata
        context = f"This painting is titled '{row['title']}', created around {row['inception']}. It belongs to the {row['movement']} movement and {row['genre']} genre. "
    
    if is_boolean:
        return f"{context}Answer with '1' or '0': {question}\nAnswer: "
    return f"{context}{question}{_EXTRACT_SUFFIX}\nAnswer: "


## Step 5 — Load Queries

In [ ]:
import yaml, pandas as pd
with open("/content/kv-compression-benchmark/benchmarks/artwork_eval/configs/image_queries.yaml") as f:
    q_cfg = yaml.safe_load(f)["artwork"]
queries = [(q, True) for q in q_cfg["filter_queries"]] + [(q, False) for q in q_cfg["extract_queries"]]

df = pd.read_csv(DATASET_PATH).head(MAX_ROWS)
df["record_id"] = range(len(df))
# Helper to map image tail to local path
def get_ip(url):
    tail = url.split("/")[-1].split("?")[0]
    return str(IMAGES_DIR / tail)
df["image_path"] = df["image_url"].apply(get_ip)


## Step 6 — Multi-Method Inference with Checkpoint

In [ ]:
import csv, gc, torch, os
from PIL import Image

def run_gen(image_path, prompt, PressCls, ratio):
    press = PressCls(compression_ratio=float(ratio))
    if hasattr(press, "update_model_and_tokenizer"):
        press.update_model_and_tokenizer(model, processor.tokenizer)
    image = Image.open(image_path).convert("RGB")
    conv = [{"role":"user","content":[{"type":"image"},{"type":"text","text":prompt}]}]
    p_text = processor.apply_chat_template(conv, add_generation_prompt=True)
    inputs = processor(images=image, text=p_text, return_tensors="pt").to(model.device)
    with torch.no_grad(), press(model):
        out = model.generate(**inputs, max_new_tokens=50, pad_token_id=processor.tokenizer.pad_token_id)
    return processor.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

# Checkpoint
def load_done():
    if not os.path.exists(CHECKPOINT_PATH): return set()
    d = pd.read_csv(CHECKPOINT_PATH)
    return set(zip(d.method, d.ratio.map(str), d.record_id, d.query))

done = load_done()
f = open(CHECKPOINT_PATH, "a", newline="", encoding="utf-8")
writer = csv.writer(f)
if os.path.getsize(CHECKPOINT_PATH) == 0:
    writer.writerow(["method","ratio","record_id","query","answer","error"])

for m_name, (Cls, use_cpt) in EXPERIMENT_CONFIGS.items():
    for r in COMPRESSION_RATIOS:
        for _, row in df.iterrows():
            for q_text, is_bool in queries:
                if (m_name, str(float(r)), row.record_id, q_text) in done: continue
                
                prompt = build_prompt(row, q_text, is_bool, use_cpt)
                ans = err = ""
                try:
                    ans = run_gen(row.image_path, prompt, Cls, r)
                except Exception as e:
                    err = str(e)[:200]
                    print(f"Err {m_name} R{r} ID{row.record_id}: {err}")
                
                writer.writerow([m_name, r, row.record_id, q_text, ans, err])
                f.flush()
                gc.collect(); torch.cuda.empty_cache()
f.close()
print("All Done!")


## Step 7-8 — Evaluate

In [ ]:
# Reconstruct results tree and run Step 8 Evaluator
import pandas as pd
from pathlib import Path
ck = pd.read_csv(CHECKPOINT_PATH)
ck = ck[ck.error.isna()]

for (m, r), g in ck.groupby(["method", "ratio"]):
    d = RESULTS_ROOT / "artwork" / f"Llava3-8b-{m}" / f"{float(r):.2f}"
    d.mkdir(parents=True, exist_ok=True)
    g[["record_id","query","answer"]].to_csv(d / "results.csv", index=False)

import sys
sys.path.append("/content/kv-compression-benchmark/benchmarks/artwork_eval")
from evaluation.evaluator import EvaluationManager
mgr = EvaluationManager(config_path="/content/kv-compression-benchmark/benchmarks/artwork_eval/evaluation/evaluation_config.yaml", results_dir=str(RESULTS_ROOT))
res = mgr.evaluate_all()
display(res.groupby(["model_tag","ratio"])[["precision","recall","f1"]].mean())
